# Get data for model training

In [89]:
import pandas as pd
from pathlib import Path
import cv2
import sys
sys.path.append("../src/")
import json
from helper_functions.image_functions import show_image
import os

In [ ]:
# with open("/home/mellanie/Desktop/Tristan_2026_code/detectron2/model2_50%_col0_data/train.json") as f: train = json.load(f)
# # with open("/home/mellanie/Desktop/Tristan_2026_code/detectron2/train.json") as f: test = json.load(f)

# train_names = {img["file_name"] for img in train["images"]}
# # test_names = {img["file_name"] for img in test["images"]}

# # print(f"Overlap: {len(train_names & test_names)}")

# print(len(train_names))
# # print(len(test_names))



61


In [11]:
path = "/home/trist/projects/Plating_2026-06-18.xlsx"
df = pd.read_excel(path)

# Save to CSV without the pandas row index numbers
df.to_csv("/home/trist/projects/output.csv", index=False, encoding="utf-8")

In [62]:
SPLIT = 3

# None = drop the shape entirely, otherwise rename to root_hair
LABEL_MAP = {
    "root": None,
    "root_hair": "root_hair",
    "bump": "root_hair",
    "background_root_hair": "root_hair",
    "tangled_root_hair": "root_hair",
    "edge_root_hair": "root_hair",
    "embedded_root_hair": "root_hair",
    "bubble": None,
    "dirt": None,
}

CATEGORIES = [{"id": 0, "name": "root_hair"}]
CATEGORY_NAME_TO_ID = {c["name"]: c["id"] for c in CATEGORIES}

In [ ]:

def get_pairings(filtered):

    images = []
    # jsons = []

    path_stub = "/home/trist/projects/Annotated_dataset/"
    index = 0
    for _, seed_log in filtered.iterrows():
        date = int(seed_log["Date"])
        nitrogen = seed_log["Nitrate concentration"]
        plate = int(seed_log["Plate"])
        number = int(seed_log["Image number\non the plate "])
        
        name = path_stub + f"{date}_{nitrogen}_Plate{plate}_DAG6_1000_L/{number:06d}.tif"

        # these are the names of two folders. They are incorrectly named
        oldpath = path_stub + f"{date}_{nitrogen}_Plate{plate}_DAG6_1000_L/labelme_annotation/"
        oldpath2 = path_stub + f"{date}_{nitrogen}_Plate{plate}_DAG6_1000_L/labelme_annotatios/"

        newpath = path_stub + f"{date}_{nitrogen}_Plate{plate}_DAG6_1000_L/labelme_annotations/"

        if os.path.isdir(oldpath) and not os.path.exists(newpath):
            print(f"fixing {oldpath} to {newpath}")
            os.rename(oldpath, newpath)

        if os.path.isdir(oldpath2) and not os.path.exists(newpath):
            print(f"fixing {oldpath2} to {newpath}")
            os.rename(oldpath2, newpath)
        
        label= path_stub + f"{date}_{nitrogen}_Plate{plate}_DAG6_1000_L/labelme_annotations/{number:06d}.json"

        images.append((name, label, seed_log["Ecotype"]))

        index += 1

        

    return images

In [64]:

def labelme_to_coco_single_class(pairs, label_map, category_name_to_id, out_path):
    skipped = 0

    coco = {
        "images": [],
        "annotations": [],
        "categories": [{"id": v, "name": k} for k, v in category_name_to_id.items()],
    }

    ann_id = 0
    for img_id, (img_path, json_path, ecotype) in enumerate(pairs):

        with open(json_path) as f:
            lm = json.load(f)


        coco["images"].append({
            "id": img_id,
            "file_name": img_path,
            "width": lm["imageWidth"],
            "height": lm["imageHeight"],
            "ecotype": ecotype
        })

        for shape in lm["shapes"]:
            orig_label = shape["label"]
            
            if orig_label not in label_map:

                if orig_label == "empty":
                    new_label = None

                elif orig_label in ["edge__root_hair", "tangled", "tangled__root_hair"]:
                    new_label = "root_hair"

                else:
                    raise ValueError(f"Unmapped label '{orig_label}' in {json_path}")


            else:
                new_label = label_map[orig_label]

            if new_label is None:
                continue

            points = shape["points"]
            xs = [p[0] for p in points]
            ys = [p[1] for p in points]
            x_min, y_min = min(xs), min(ys)
            w, h = max(xs) - x_min, max(ys) - y_min

            segmentation = [[coord for point in points for coord in point]]

            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": category_name_to_id[new_label], # single class
                "old_category": orig_label, # multuclass
                "segmentation": segmentation,
                "bbox": [x_min, y_min, w, h],
                "area": w * h,
                "iscrowd": 0,
            })
            ann_id += 1

    with open(out_path, "w") as f:
        json.dump(coco, f, indent=2)

    print(skipped, "skipped")

In [65]:
dataset_info = "/home/trist/projects/Plating_2026-06-18.csv"
data = pd.read_csv(dataset_info, encoding="cp1252")

In [66]:
data.columns

Index(['Date', 'Nitrate concentration', 'Plate', 'Mode', 'Ecotype',
       'Seed number for the specific concentration', 'Annotation', 'Annotator',
       'Seed number on the plate ', 'Image number\non the plate ', 'Stitched',
       'reason for unusability of images ', 'Comment', 'Split_1', 'Split_2',
       'Split_3'],
      dtype='str')

# Get Training Data

In [67]:
# All of these will be loaded and used for the training data
training = data[data[f"Split_{SPLIT}"] == "training"]
training = training[training["Annotation"] == "yes"]
training.head()

,Date,Nitrate concentration,Plate,Mode,Ecotype,Seed number for the specific concentration,Annotation,Annotator,Seed number on the plate,Image number\non the plate,Stitched,reason for unusability of images,Comment,Split_1,Split_2,Split_3
4,20240918.0,N4,1.0,L,Pu2-23,_004,yes,Roberto Zarazua,5.0,5.0,no_need,NaN,NaN,testing,testing,training
5,20240918.0,N4,1.0,L,Pu2-23,_005,yes,Roberto Zarazua,6.0,6.0,no_need,NaN,NaN,testing,testing,training
15,20240918.0,N4,1.0,L,Shahdara,_004,yes,Roberto Zarazua,16.0,16.0,no_need,NaN,NaN,testing,testing,training
16,20240918.0,N4,1.0,L,Shahdara,_005,yes,Roberto Zarazua,17.0,17.0,no_need,NaN,NaN,testing,testing,training
17,20240918.0,N4,1.0,L,Shahdara,_006,yes,Roberto Zarazua,18.0,18.0,no_need,NaN,NaN,testing,testing,training


In [68]:
# Look at shape
training.shape

(462, 16)

In [69]:
# see distribution of nitrogen
training["Nitrate concentration"].value_counts()

Nitrate concentration
N4      225
N16      67
N32      67
N64      54
N128     49
Name: count, dtype: int64

In [71]:
# 23 from last time I was in Germany
# GENOTYPES = ["Pu2-23",
#              "Shahdara",
#              "Ws-0",
#              "St-0",
#              "Oy-0",
#              "Uk-1",
#              "Bay-0",
#              "Kondara",
#              "No-0",
#              "Hs-0",
#              "Cvi-0",
#              "Lip-0",
#              "id2-3",
#              "pme38-1",
#              "Ge-0",
#              "Ting-1",
#              "Ts-1",
#              "Col-0*"]

# GENOTYPES = [  'Pu2-23', 'Shahdara',     'Ws-0',     'St-0',     'Uk-1',     'Oy-0',
#   'Kondara',    'Bay-0',    'Col-0',     'Hs-0',    'Cvi-0',     'No-0',
#      'Nd-1',     'Wa-1',    'Pro-0',    'Lip-0',   'Ting-1',     'Ge-0',
#     'id2-3',  'pme38-1',   'Col-0*',     'Zu-1',    'Lu4-2',   'UKID93',
#   'Tscha-1',    'Kil-0', 'IP-Cmo-3',  'Dospa-1',      'Hod']

GENOTYPES = [  'Pu2-23', 'Shahdara',     'Ws-0',     'St-0',     'Uk-1',     'Oy-0',
  'Kondara',    'Bay-0',    'Col-0',     'Hs-0',    'Cvi-0',     'No-0',
      'Lip-0',   'Ting-1',     'Ge-0',
    'id2-3',  'pme38-1',   'Col-0*',     'Zu-1',    'Lu4-2',   'UKID93',
  'Tscha-1',    'Kil-0', 'IP-Cmo-3',  'Dospa-1']

training = training[training["Ecotype"].isin(GENOTYPES)]

In [73]:
training["Ecotype"].unique()

<StringArray>
[  'Pu2-23', 'Shahdara',     'Ws-0',     'St-0',     'Uk-1',     'Oy-0',
  'Kondara',    'Bay-0',    'Col-0',     'Hs-0',    'Cvi-0',     'No-0',
    'Lip-0',   'Ting-1',     'Ge-0',    'id2-3',  'pme38-1',   'Col-0*',
     'Zu-1',    'Lu4-2',   'UKID93',  'Tscha-1',    'Kil-0', 'IP-Cmo-3',
  'Dospa-1']
Length: 25, dtype: str

In [77]:
# see distribution of ecotypes
training = training[(training["Annotation"] == "yes") | (training["Annotation"] == "yes ")]

training["Ecotype"].value_counts()

Ecotype
Col-0*      128
id2-3       125
Shahdara     25
Ws-0         19
Pu2-23       17
St-0          7
Uk-1          7
Oy-0          7
Kondara       7
Bay-0         7
Col-0         7
Hs-0          7
Cvi-0         7
No-0          7
Lip-0         7
Ting-1        7
Ge-0          7
pme38-1       7
Zu-1          7
Lu4-2         7
UKID93        7
Tscha-1       7
Kil-0         7
IP-Cmo-3      7
Dospa-1       7
Name: count, dtype: int64

In [78]:
training.shape

(454, 16)

Optional Random drop

In [ ]:
# genotype = "Col-0*"  

# # Get indices of rows matching that genotype
# genotype_rows = training[training["Ecotype"] == genotype]

# # Randomly select 25% of those rows to drop
# rows_to_drop = genotype_rows.sample(frac=0.50, random_state=42).index

# # Drop them from the full dataframe
# training = training.drop(rows_to_drop) 

In [104]:
# training["Ecotype"].value_counts()

In [105]:
# training["Ecotype"].value_counts().plot.pie(autopct='%1.1f%%', ylabel='', figsize=(6, 6))

In [106]:
# col0 = training[training["Ecotype"].isin(["Col-0*"])]

# col0_images = get_pairings(col0)


# # # generate cocos
# labelme_to_coco_single_class(col0_images, LABEL_MAP, CATEGORY_NAME_TO_ID, "col0_of_2.json")

# Get Evaluation Data

this will need to be updated to fit the new genotypes when they come

In [79]:
# Genotypes we have at the moment
# GENOTYPES = ["Pu2-23",
#              "Shahdara",
#              "Ws-0",
#              "St-0",
#              "Oy-0",
#              "Uk-1",
#              "Bay-0",
#              "Kondara",
#              "No-0",
#              "Hs-0",
#              "Cvi-0",
#              "Lip-0",
#              "id2-3",
#              "pme38-1",
#              "Ge-0",
#              "Ting-1",
#              "Ts-1",
#              "Col-0*"]

GENOTYPES = [  'Pu2-23', 'Shahdara',     'Ws-0',     'St-0',     'Uk-1',     'Oy-0',
  'Kondara',    'Bay-0',    'Col-0',     'Hs-0',    'Cvi-0',     'No-0',
      'Lip-0',   'Ting-1',     'Ge-0',
    'id2-3',  'pme38-1',   'Col-0*',     'Zu-1',    'Lu4-2',   'UKID93',
  'Tscha-1',    'Kil-0', 'IP-Cmo-3',  'Dospa-1']


# GENOTYPES = ["Ws-0", "Shahdara"]

In [80]:
evaluation = data[data[f"Split_{SPLIT}"] == "testing"]

In [81]:
evaluation["Nitrate concentration"].value_counts()

Nitrate concentration
N4    105
Name: count, dtype: int64

In [82]:
evaluation = evaluation[evaluation["Ecotype"].isin(GENOTYPES)]


# for eco in evaluation["Ecotype"].unique():
#     if eco in GENOTYPES:
#         print(eco)
#         continue

#     count = (evaluation["Ecotype"] == eco).sum()
#     if count >= 10:
#         print("ecotype not listed", eco)

evaluation = evaluation[(evaluation["Annotation"] == "yes") | (evaluation["Annotation"] == "yes ")]

# evaluation["Ecotype"].value_counts()

In [83]:
evaluation.shape


(75, 16)

In [84]:
evaluation["Ecotype"].value_counts()

Ecotype
Pu2-23      3
Ws-0        3
Shahdara    3
St-0        3
Uk-1        3
Oy-0        3
Kondara     3
Col-0       3
Bay-0       3
Hs-0        3
Cvi-0       3
No-0        3
Lip-0       3
Ting-1      3
Ge-0        3
id2-3       3
pme38-1     3
Col-0*      3
Lu4-2       3
UKID93      3
Zu-1        3
Tscha-1     3
IP-Cmo-3    3
Kil-0       3
Dospa-1     3
Name: count, dtype: int64

In [113]:
# evaluation_images = get_pairings(evaluation)
# evaluation_images

# Get data into json

In [103]:
# Pair images with jsons
training_images = get_pairings(training)
evaluation_images = get_pairings(evaluation)


# generate cocos
labelme_to_coco_single_class(training_images, LABEL_MAP, CATEGORY_NAME_TO_ID, "train.json")
labelme_to_coco_single_class(evaluation_images, LABEL_MAP, CATEGORY_NAME_TO_ID, "test.json")

fixing /home/trist/projects/Annotated_dataset/20260311_N4_Plate6_DAG6_1000_L/labelme_annotatios/ to /home/trist/projects/Annotated_dataset/20260311_N4_Plate6_DAG6_1000_L/labelme_annotations/


FileNotFoundError: [Errno 2] No such file or directory: '/home/trist/projects/Annotated_dataset/20260318_N4_Plate1_DAG6_1000_L/labelme_annotations/000025.json'

In [115]:
# evaluation_images